# analysis_b / 03b — Latent Topic Discovery (GSDMM)

**Why GSDMM instead of LDA?**

LDA models every document as a *mixture* of topics, which fits long documents well but is a poor assumption for short posts: a single Reddit post almost always orbits one idea. GSDMM (Yin & Wang 2014, *Dirichlet Multinomial Mixture*) assigns each document to **exactly one topic**, which matches that reality and tends to produce sharper, more interpretable clusters on short-text corpora.

Additional advantages:
- K is an **upper bound**, not a fixed count — empty clusters are pruned naturally.
- No train/test split needed for coherence; perplexity is replaced by PMI + cluster utilisation.
- Hard assignments make downstream grouping analysis trivial.

**Pipeline:** vocab + token lists → K sweep (utilisation + PMI coherence) → fit final model → topic profiling (top words, FREX, top docs) → score comparison → write artifacts

**Reads:** `artifacts/posts_clean.parquet`  
**Writes:** `artifacts/theta_gsdmm.npy`, `artifacts/phi_gsdmm.npy`, `artifacts/gsdmm_vocab.json`, `artifacts/topic_labels_gsdmm.json`

> **Install once:** `pip install gsdmm` (or `pip install git+https://github.com/rwalk/gsdmm.git`)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import textwrap
import warnings
import itertools
from pathlib import Path
from collections import defaultdict
from scipy.sparse import csc_matrix
from scipy.stats import spearmanr, kruskal, mannwhitneyu
from sklearn.feature_extraction.text import CountVectorizer
from gsdmm import MovieGroupProcess

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

ARTIFACTS = Path('artifacts')
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

df = pd.read_parquet(ARTIFACTS / 'posts_clean.parquet')
print(f'Posts: {len(df):,}')
df[['text_lemma', 'score', 'year_month']].head(3)

ModuleNotFoundError: No module named 'gsdmm'

## 1. Vocabulary & Token Lists

GSDMM expects each document as a **list of unique tokens** (no multiplicity). We reuse the same CountVectorizer filters as LDA (`min_df=10`, `max_df=0.90`) so results are directly comparable, then convert each doc to the subset of its tokens that fall in-vocabulary.

In [ ]:
vec = CountVectorizer(min_df=10, max_df=0.90)
vec.fit(df['text_lemma'])
vocab_set = set(vec.vocabulary_.keys())
vocab_list = vec.get_feature_names_out().tolist()
vocab_to_idx = {w: i for i, w in enumerate(vocab_list)}
V = len(vocab_list)

print(f'Vocabulary size : {V:,} terms')

# Convert each doc to a deduplicated list of in-vocab tokens
def doc_to_tokens(text):
    return list({w for w in str(text).split() if w in vocab_set})

docs = df['text_lemma'].map(doc_to_tokens).tolist()

lengths = [len(d) for d in docs]
print(f'Docs            : {len(docs):,}')
print(f'Tokens/doc      : min={min(lengths)}, median={int(np.median(lengths))}, max={max(lengths)}')

# GSDMM cannot handle empty docs — flag and count them
empty_mask = np.array([len(d) == 0 for d in docs])
print(f'Empty docs      : {empty_mask.sum():,}  (will be assigned cluster -1)')

## 2. K Selection

Sweep K (upper bound). For each value we record:
- **Active clusters** — clusters with at least one document. Ideally stabilises before hitting K.
- **PMI coherence** — same pairwise PMI as 03_lda, averaged over active topics.

Choose K where active clusters plateau and coherence is highest.

In [ ]:
# Binary DTM for PMI coherence (reuse same vocab)
dtm = vec.transform(df['text_lemma'])
dtm_bin = csc_matrix((dtm > 0).astype(np.float32))

def pmi_coherence(topic_words_idx, dtm_csc, n_top=10):
    top_w = topic_words_idx[:n_top]
    cols = [
        np.asarray(dtm_csc.getcol(int(w)).todense()).flatten() > 0
        for w in top_w
    ]
    pmis = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            p_i  = cols[i].mean()
            p_j  = cols[j].mean()
            p_ij = (cols[i] & cols[j]).mean()
            if p_i > 0 and p_j > 0 and p_ij > 0:
                pmis.append(np.log(p_ij / (p_i * p_j)))
    return float(np.mean(pmis)) if pmis else 0.0


def build_phi_from_mgp(mgp, vocab_to_idx, V):
    """Extract (K, V) topic-word matrix from a fitted MovieGroupProcess."""
    phi = np.zeros((mgp.K, V), dtype=np.float32)
    for k in range(mgp.K):
        for word, count in mgp.cluster_word_distribution[k].items():
            if word in vocab_to_idx:
                phi[k, vocab_to_idx[word]] = count
    row_sums = phi.sum(axis=1, keepdims=True)
    phi = np.where(row_sums > 0, phi / row_sums, 0.0)
    return phi

In [ ]:
non_empty_docs = [d for d in docs if d]   # exclude empty docs for fitting

K_RANGE = [4, 6, 8, 10, 12, 14, 16, 18, 20]
results = []

for K_upper in K_RANGE:
    mgp_k = MovieGroupProcess(K=K_upper, alpha=0.1, beta=0.1, n_iters=30)
    y_k   = mgp_k.fit(non_empty_docs)

    counts    = np.array(mgp_k.cluster_doc_count)
    active    = int((counts > 0).sum())
    phi_k     = build_phi_from_mgp(mgp_k, vocab_to_idx, V)

    # PMI coherence only over active topics
    active_idx = np.where(counts > 0)[0]
    coh_vals   = []
    for k in active_idx:
        top_idx = phi_k[k].argsort()[::-1][:10]
        coh_vals.append(pmi_coherence(top_idx, dtm_bin))
    coh = float(np.mean(coh_vals)) if coh_vals else 0.0

    results.append({'K_upper': K_upper, 'active': active, 'coherence': coh})
    print(f'K_upper={K_upper:2d}  active={active:2d}  coherence={coh:.4f}')

results_df = pd.DataFrame(results)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(results_df['K_upper'], results_df['active'], marker='o', color='steelblue')
ax1.set_title('Active clusters vs K upper bound')
ax1.set_xlabel('K (upper bound)')
ax1.set_ylabel('Non-empty clusters')

ax2.plot(results_df['K_upper'], results_df['coherence'], marker='o', color='steelblue')
ax2.set_title('Mean PMI Coherence (active topics only)')
ax2.set_xlabel('K (upper bound)')
ax2.set_ylabel('Mean pairwise PMI (higher = better)')

plt.suptitle('K selection — pick K where active clusters plateau and coherence peaks')
plt.tight_layout()
plt.show()

## 3. Fit Final Model

Set `K_UPPER` based on the plots above. Rule of thumb: choose where active clusters stop growing — the algorithm is already converging to that many natural topics.

In [ ]:
K_UPPER = 10   # ← adjust after inspecting the plots

mgp = MovieGroupProcess(K=K_UPPER, alpha=0.1, beta=0.1, n_iters=50)
y_non_empty = mgp.fit(non_empty_docs)

counts = np.array(mgp.cluster_doc_count)
active_clusters = np.where(counts > 0)[0]
K_ACTIVE = len(active_clusters)
print(f'K_upper={K_UPPER}  |  active clusters={K_ACTIVE}')
print(f'Docs per active cluster:')
for k in active_clusters:
    print(f'  Cluster {k:2d}: {counts[k]:,} docs ({counts[k]/len(non_empty_docs)*100:.1f}%)')

In [ ]:
# Rebuild full assignment array over all docs (including empty ones → -1)
y_all = np.full(len(docs), -1, dtype=int)
non_empty_idx = [i for i, d in enumerate(docs) if d]
for i, k in zip(non_empty_idx, y_non_empty):
    y_all[i] = k

# phi: (K_UPPER, V) topic-word matrix
phi = build_phi_from_mgp(mgp, vocab_to_idx, V)
print(f'phi shape : {phi.shape}')

# theta: (N, K_UPPER) one-hot assignment matrix
theta = np.zeros((len(docs), K_UPPER), dtype=np.float32)
for i, k in enumerate(y_all):
    if k >= 0:
        theta[i, k] = 1.0
print(f'theta shape : {theta.shape}')
print(f'theta row-sum check (non-empty docs, should be 1.0): {theta[~empty_mask].sum(axis=1).mean():.4f}')

## 4. Topic Profiling

For each **active** cluster: top words (raw + FREX) and a sample of representative documents. Read the documents — words alone are often misleading.

In [ ]:
N_TOP_WORDS = 20

print('Top words per active topic')
print('=' * 70)
for k in active_clusters:
    top_idx   = phi[k].argsort()[::-1][:N_TOP_WORDS]
    top_words = ', '.join(vocab_list[i] for i in top_idx)
    print(f'Topic {k:2d} ({counts[k]:,} docs): {top_words}')

In [ ]:
# Pairwise top-word overlap across active clusters
N_TOP = 20
top_word_sets = {}
for k in active_clusters:
    top_idx = phi[k].argsort()[::-1][:N_TOP]
    top_word_sets[k] = set(vocab_list[i] for i in top_idx)

n = K_ACTIVE
overlap = np.zeros((n, n), dtype=int)
labels  = [f'T{k}' for k in active_clusters]
for ii, ki in enumerate(active_clusters):
    for jj, kj in enumerate(active_clusters):
        overlap[ii, jj] = len(top_word_sets[ki] & top_word_sets[kj])

fig, ax = plt.subplots(figsize=(max(5, n), max(4, n - 1)))
sns.heatmap(overlap, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title(f'Shared top-{N_TOP} words between topic pairs')
plt.tight_layout()
plt.show()

print(f'\nShared words (top-{N_TOP}) per pair:')
for ii, ki in enumerate(active_clusters):
    for jj, kj in enumerate(active_clusters):
        if ii >= jj:
            continue
        shared = sorted(top_word_sets[ki] & top_word_sets[kj])
        print(f'  T{ki} ∩ T{kj} ({len(shared)}): {', '.join(shared) if shared else "none"}')

### FREX — Frequency × Exclusivity

Re-ranks words by the harmonic mean of ECDF-normalised frequency and exclusivity. Cuts through corpus-wide filler words that dominate raw top-word lists.

In [ ]:
from scipy.stats import rankdata

def frex(phi, w=0.5, n_top=15):
    K, V = phi.shape
    col_sums = phi.sum(axis=0)
    excl = phi / np.where(col_sums > 0, col_sums, 1)
    freq_ecdf = np.apply_along_axis(lambda row: rankdata(row) / V, axis=1, arr=phi)
    excl_ecdf = np.apply_along_axis(lambda row: rankdata(row) / V, axis=1, arr=excl)
    scores = 1.0 / (w / freq_ecdf + (1 - w) / excl_ecdf)
    top_idx = scores.argsort(axis=1)[:, ::-1][:, :n_top]
    return scores, top_idx

N_FREX = 15
frex_scores, frex_top = frex(phi, w=0.5, n_top=N_FREX)

print(f'FREX top-{N_FREX} words per active topic')
print('=' * 70)
for k in active_clusters:
    words = ', '.join(vocab_list[i] for i in frex_top[k])
    print(f'Topic {k:2d}: {words}')

In [ ]:
# Side-by-side raw vs FREX comparison
N_CMP = 10
data = {}
for k in active_clusters:
    raw_words  = [vocab_list[i] for i in phi[k].argsort()[::-1][:N_CMP]]
    frex_words = [vocab_list[i] for i in frex_top[k][:N_CMP]]
    data[(f'Topic {k}', 'Raw')]  = raw_words
    data[(f'Topic {k}', 'FREX')] = frex_words

cols   = pd.MultiIndex.from_tuples(data.keys())
cmp_df = pd.DataFrame(data, index=range(1, N_CMP + 1))
cmp_df.index.name = 'Rank'
cmp_df

### Document inspection

In [ ]:
N_TOP_DOCS = 10
W = 80

for k in active_clusters:
    print('=' * W)
    print(f'TOPIC {k} — top {N_TOP_DOCS} documents by theta loading')
    print('=' * W)
    top_doc_idx = theta[:, k].argsort()[::-1][:N_TOP_DOCS]
    for rank, idx in enumerate(top_doc_idx, start=1):
        row  = df.iloc[idx]
        date = str(row.get('date', row.get('year_month', '')))
        print(f'  [{rank}] {date} | score={row["score"]}')
        print(textwrap.fill(str(row['text_clean']), width=W - 4,
                            initial_indent='      ', subsequent_indent='      '))
        print()
    print()

In [ ]:
# Sampled docs — dominant-cluster members (posts only)
N_SAMPLE = 5
is_post  = (df['type'] == 'post').values if 'type' in df.columns else np.ones(len(df), dtype=bool)

for k in active_clusters:
    print('\n' + '─' * W)
    print(f'  TOPIC {k} — {N_SAMPLE} sampled posts (dominant-cluster members)')
    print('─' * W)
    members    = np.where((y_all == k) & is_post)[0]
    sample_idx = rng.choice(members, size=min(N_SAMPLE, len(members)), replace=False)
    for rank, idx in enumerate(sample_idx, start=1):
        row  = df.iloc[idx]
        date = str(row.get('date', row.get('year_month', '')))
        print(f'\n  [{rank}]  {date}  |  score={row["score"]}')
        print('  ' + '─' * (W - 2))
        print(textwrap.fill(str(row['text_clean']), width=W - 4,
                            initial_indent='  ', subsequent_indent='  '))

## 5. Cluster Size Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(max(7, K_ACTIVE + 2), 4))
labels_active = [f'T{k}' for k in active_clusters]
sizes_active  = [counts[k] for k in active_clusters]

bars = ax.bar(labels_active, sizes_active, color='steelblue')
ax.bar_label(bars, labels=[f'{s:,}' for s in sizes_active], padding=3, fontsize=9)
ax.set_title('Document count per active GSDMM cluster')
ax.set_ylabel('# documents')
ax.set_xlabel('Topic')
plt.tight_layout()
plt.show()

## 6. Score Comparison Across Topics

Because GSDMM gives **hard assignments**, the theta column is binary (0/1) — Spearman on a binary column is technically valid but uninformative. Instead we compare the score distributions across topic groups using:

- **Kruskal-Wallis** — non-parametric omnibus test (are all groups from the same distribution?)
- **Pairwise Mann-Whitney U** with Bonferroni correction — which pairs differ?

In [ ]:
score_vals = df['score'].fillna(0).values

# Groups: one list of scores per active cluster (exclude empty-doc rows)
groups = {k: score_vals[y_all == k] for k in active_clusters}

# Kruskal-Wallis
kw_stat, kw_p = kruskal(*groups.values())
print(f'Kruskal-Wallis H={kw_stat:.2f}  p={kw_p:.4e}\n')

# Median scores per topic
print('Median score per topic:')
for k in active_clusters:
    g = groups[k]
    print(f'  Topic {k:2d}: median={np.median(g):.1f}  mean={g.mean():.2f}  n={len(g):,}')

# Pairwise Mann-Whitney with Bonferroni correction
pairs = list(itertools.combinations(active_clusters, 2))
alpha_bonf = 0.05 / len(pairs)
print(f'\nPairwise Mann-Whitney U  (Bonferroni α={alpha_bonf:.4f}):')
for ki, kj in pairs:
    stat, p = mannwhitneyu(groups[ki], groups[kj], alternative='two-sided')
    sig = '*' if p < alpha_bonf else ''
    print(f'  T{ki} vs T{kj}: U={stat:.0f}  p={p:.4e}  {sig}')

In [ ]:
# Box-plot: score distribution per active topic
plot_data = pd.DataFrame({
    'score': score_vals,
    'topic': [f'T{k}' if k >= 0 else 'empty' for k in y_all]
})
plot_data = plot_data[plot_data['topic'] != 'empty']
topic_order = [f'T{k}' for k in active_clusters]

fig, ax = plt.subplots(figsize=(max(8, K_ACTIVE + 2), 5))
sns.boxplot(
    data=plot_data, x='topic', y='score', order=topic_order,
    showfliers=False, palette='muted', ax=ax
)
ax.set_title('Score distribution per GSDMM topic (outliers hidden)')
ax.set_xlabel('Topic')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

## 7. Write Artifacts

Artifacts use a `_gsdmm` suffix so they coexist with the LDA outputs. Downstream notebooks that depend on `theta.npy` / `phi.npy` can be pointed here instead.

In [ ]:
np.save(ARTIFACTS / 'theta_gsdmm.npy', theta)
np.save(ARTIFACTS / 'phi_gsdmm.npy',   phi)

with open(ARTIFACTS / 'gsdmm_vocab.json', 'w') as fh:
    json.dump(vocab_list, fh)

# Cluster assignments (integer per doc, -1 for empty docs)
np.save(ARTIFACTS / 'gsdmm_assignments.npy', y_all)

# Label template — fill in before running temporal analysis
template = {str(k): f'Topic {k} — label here' for k in active_clusters.tolist()}
with open(ARTIFACTS / 'topic_labels_gsdmm.json', 'w') as fh:
    json.dump(template, fh, indent=2)

print('Artifacts written:')
for name in ('theta_gsdmm.npy', 'phi_gsdmm.npy', 'gsdmm_vocab.json',
             'gsdmm_assignments.npy', 'topic_labels_gsdmm.json'):
    p = ARTIFACTS / name
    if p.exists():
        print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')

print()
print(f'Active clusters : {K_ACTIVE}')
print(f'K upper bound   : {K_UPPER}')
print()
print('Next: fill in topic_labels_gsdmm.json, then run temporal analysis.')